In [ ]:
# Keras 사용 모델 학습 코드

import tensorflow as tf
from tensorflow.keras import layers, models, optimizers
from tensorflow.keras.applications import ResNet50 # Keras 공식 지원은 ResNet50부터입니다.
from tensorflow.keras.callbacks import EarlyStopping
# ResNet18을 꼭 쓰고 싶다면 분류 라이브러리인 classification_models를 쓰거나 ResNet50을 추천합니다.
import numpy as np
from sklearn.metrics import classification_report

print("🔥 현재 인식된 GPU 목록:", tf.config.list_physical_devices('GPU'))

# =========================================================
# 1. 데이터 전처리 및 로드 (ImageFolder 대신 image_dataset_from_directory)
# =========================================================
img_height, img_width = 256, 256
batch_size = 32

# 데이터셋을 자동으로 Train/Validation으로 나누어 로드합니다.
train_ds = tf.keras.utils.image_dataset_from_directory(
    './my_room_dataset',
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=(img_height, img_width),
    batch_size=batch_size
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    './my_room_dataset',
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=(img_height, img_width),
    batch_size=batch_size
)

# ResNet 입력 규격에 맞는 전처리 (0~255 -> -1~1 혹은 ResNet 표준화)
normalization_layer = layers.Rescaling(1./255) # 혹은 ResNet50 전용 preprocess_input 사용
train_ds = train_ds.map(lambda x, y: (tf.keras.applications.resnet50.preprocess_input(x), y))
val_ds = val_ds.map(lambda x, y: (tf.keras.applications.resnet50.preprocess_input(x), y))

# =========================================================
# 2. 모델 정의 (ResNet50 전이 학습)
# =========================================================
# Keras 공식 라이브러리는 ResNet18 대신 ResNet50이 기본입니다. (더 깊고 정확함)
base_model = tf.keras.applications.ResNet50(weights='imagenet', include_top=False, input_shape=(256, 256, 3))

# 베이스 모델 가중치는 고정 (Freezing) - 학습되지 않게 함
base_model.trainable = False 

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(1, activation='sigmoid') # ⭐️ 수정 1: 클래스가 2개(이진 분류)일 때는 출력 노드를 1개로 하고 sigmoid를 씁니다!
])

# =========================================================
# 3. 컴파일 및 학습
# =========================================================
model.compile(
    optimizer=optimizers.Adam(learning_rate=0.0001),
    loss='binary_crossentropy', # ⭐️ 수정 2: 이진 분류 전용 손실 함수로 변경합니다!
    metrics=['accuracy']
)

# ⭐️ Early Stopping 콜백 정의
early_stopping = EarlyStopping(
    monitor='val_loss',        # 감시할 지표 (검증 세트의 오차)
    patience=3,                # 오차가 줄어들지 않아도 몇 번의 에폭(Epoch)을 더 참을 것인가?
    restore_best_weights=True  # 멈췄을 때, 가장 성능이 좋았던 시점의 가중치로 자동 복구! (매우 중요)
)

print("\n🚀 Keras ResNet 전이 학습을 시작합니다!")
epochs = 200
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=epochs,
    callbacks=[early_stopping]
)

# =========================================================
# 4. 모델 저장 (keras 형식)
# =========================================================
model.save('resnet_room_clean_model.keras')
print("✨ Keras 모델 학습 및 저장(resnet_room_clean_model.keras) 완료!")

print("\n📊 검증 데이터(Validation)를 바탕으로 정밀 평가를 시작합니다...")

y_true = []
y_pred = []

# ⭐️ 주의: 데이터 순서가 섞이지 않도록(Shuffle 방지) 
# 검증 데이터셋(val_ds)에서 정답과 예측값을 한 번의 반복문에서 동시에 뽑아냅니다.
for images, labels in val_ds:
    # 1. 실제 정답(True Labels) 저장
    y_true.extend(labels.numpy())
    
    # 2. 모델 예측 (0.0 ~ 1.0 사이의 확률값 반환)
    preds = model.predict(images, verbose=0)
    
    # 3. 확률이 0.5보다 크면 1(Dirty), 작거나 같으면 0(Clean)으로 변환
    preds_binary = (preds > 0.5).astype(int).flatten()
    y_pred.extend(preds_binary)

# 4. scikit-learn의 classification_report를 사용하여 4가지 지표 출력
target_names = ['Clean (0)', 'Dirty (1)']
report = classification_report(y_true, y_pred, target_names=target_names)

print("\n=======================================================")
print(report)
print("=======================================================")

In [ ]:
# Keras 사용 모델 검증 코드 (수정 통합본)

import os
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing import image

# =========================================================
# 1. 모델 세팅
# =========================================================
# 아까 저장하셨던 모델 이름에 맞게 확장자(.h5 또는 .keras)를 적어주세요.
model_path = 'resnet_room_clean_model.keras' 
model = tf.keras.models.load_model(model_path)

# 폴더명 알파벳 순서대로 클래스가 매핑됩니다. (c가 먼저니까 clean이 0, dirty가 1)
classes = ['clean', 'dirty'] 

# =========================================================
# 2. 테스트할 폴더 및 파일 읽기
# =========================================================
test_folder = './test_images/normal' 
image_files = [f for f in os.listdir(test_folder) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]

print(f"📂 '{test_folder}' 폴더에서 총 {len(image_files)}장의 이미지를 발견했습니다!\n")
print("=== 🤖 Keras AI 판독 시작 ===")

# =========================================================
# 3. 폴더 안의 여러 사진 연속으로 테스트하기
# =========================================================
for file_name in image_files:
    image_path = os.path.join(test_folder, file_name)
    
    try:
        # ⭐️ Keras 내장 함수로 이미지를 부르고 크기를 맞춥니다 (256x256)
        img = image.load_img(image_path, target_size=(256, 256))
        
        # 이미지를 숫자 배열(Numpy Array)로 변환
        img_array = image.img_to_array(img)
        
        # 모델은 항상 '배치(묶음)' 단위로 받기 때문에 차원을 하나 추가합니다.
        # (256, 256, 3) ➡️ (1, 256, 256, 3)
        input_tensor = np.expand_dims(img_array, axis=0)
        
        # ⭐️ 학습할 때 썼던 ResNet50 전처리 방식을 똑같이 적용!
        input_tensor = tf.keras.applications.resnet50.preprocess_input(input_tensor)

        # 모델 예측 (verbose=0으로 하면 지저분한 진행바가 숨겨집니다)
        outputs = model.predict(input_tensor, verbose=0)
        
        # ⭐️ 결과 해석 및 확신도(Confidence) 계산 (Sigmoid 이진 분류용으로 수정됨!)
        # 모델 마지막 레이어에 sigmoid를 썼기 때문에 outputs[0][0] 값이 0.0 ~ 1.0 사이의 확률입니다.
        probability = outputs[0][0]
        
        # 0.5를 기준으로 Clean(0)과 Dirty(1)을 나눕니다.
        if probability > 0.5:
            result = classes[1] # 'dirty'
            confidence = probability * 100
        else:
            result = classes[0] # 'clean'
            # 1에서 빼줘야 Clean에 대한 확신도가 됩니다.
            confidence = (1.0 - probability) * 100
            
        # 보기 좋게 한 줄씩 결과 출력
        print(f"[{file_name}] ➡️ 판독 결과: {result} ({confidence:.1f}% 확신)")
        
    except Exception as e:
        print(f"[{file_name}] ❌ 이미지 처리 중 에러 발생: {e}")

print("\n✨ 모든 사진의 판독이 완료되었습니다!")